# Lifespan Predictor — Choosing the RUL Cap for Your Decisions

`fit()` can choose the RUL cap `max_rul` from a set of candidates, and
`decision_horizon` tells it **which forecasts matter**: the remaining life
within which maintenance decisions are made. This notebook trains the
interpretable Cox model on NASA's C-MAPSS FD001 in **one SageMaker
training job**: it tunes the model, chooses the cap, trains the final
model and scores it on the test engines. Nothing is deployed, so there is
nothing to delete afterwards.

It runs end to end with no edits, on either Lifespan Predictor product,
including the **free tier**: FD001 has 100 training
engines, within its cap. See [Which product?](https://github.com/kooditurva/lifespan-predictor-sdk#which-product).
`sagemaker_coxph_demo.ipynb` introduces the Cox model itself.

## 1. Install the SDK

The client SDK is installed from its GitHub release tag. It is not
published to PyPI.

In [ ]:
# Other releases: https://github.com/kooditurva/lifespan-predictor-sdk/releases
!pip install -q "lifespan-predictor @ git+https://github.com/kooditurva/lifespan-predictor-sdk@v1.2.0.post1"

## 2. Point at the algorithm you subscribed to

The ARN below is the free tier's, in `eu-west-1`. If you subscribed to
the other product or work in another Region, use the ARN listed under
[Which product?](https://github.com/kooditurva/lifespan-predictor-sdk#which-product), or find it in the AWS Marketplace
console: **Manage subscriptions**, select the product, choose
**Continue to configuration**, and pick your Region.

The execution role is the IAM role SageMaker assumes to run your
training jobs. The SDK reads it from `SAGEMAKER_ROLE_ARN`. Outside
SageMaker Studio, set that variable before starting Jupyter; in Studio,
the cell below sets it from `get_execution_role()`.

In [ ]:
import os

import matplotlib.pyplot as plt

from lifespan_predictor_sdk import (
    CategoricalRange,
    ContinuousRange,
    SurvivalModel,
    download_nasa_dataset,
    get_nasa_test_dataframe,
    get_nasa_train_dataframe,
)

ALGORITHM_ARN = 'arn:aws:sagemaker:eu-west-1:985815980388:algorithm/free-tier-lifespan-predictor-m-61cb7b65231331099f089cbf75f4410e'

if 'SAGEMAKER_ROLE_ARN' not in os.environ:
    from sagemaker import get_execution_role

    os.environ['SAGEMAKER_ROLE_ARN'] = get_execution_role()

### Check your setup

What the SDK will use. Set `SAGEMAKER_BUCKET` to keep the data and models
in a bucket of your own; otherwise SageMaker's default bucket for your
account and Region is used, and created if needed.

In [ ]:
from importlib.metadata import version

import boto3

region = boto3.Session().region_name
arn_region = ALGORITHM_ARN.split(':')[3]

print('SDK version:        ', version('lifespan-predictor'))
print('Region:             ', region)
print('AWS_PROFILE:        ', os.environ.get('AWS_PROFILE', '(not set)'))
print('SAGEMAKER_ROLE_ARN: ', os.environ['SAGEMAKER_ROLE_ARN'])
print('SAGEMAKER_BUCKET:   ', os.environ.get(
    'SAGEMAKER_BUCKET', "(not set: SageMaker's default bucket)"))
print('Algorithm ARN:      ', ALGORITHM_ARN)

if region != arn_region:
    print(f'Warning: the session is in {region}, but the algorithm is in '
          f'{arn_region}. Use the ARN for your Region.')

In [ ]:
SAGEMAKER = {
    'algorithm_arn': ALGORITHM_ARN,
    'instance_type': 'ml.m5.large',
}

## 3. Load the NASA turbofan data

FD001: simulated engines, each observed once per flight cycle until it
fails. The test engines' histories stop before failure, and
`test_labels_df` holds their true remaining life, one row per engine.

In [ ]:
data_dir = 'data/nasa'
download_nasa_dataset(data_dir, datasets=[1])

train_df = get_nasa_train_dataframe(data_dir, datasets=[1])
test_df, test_labels_df = get_nasa_test_dataframe(data_dir, datasets=[1])

print(f"Training: {train_df['unit'].nunique()} engines, {len(train_df)} rows")
print(f"Test:     {test_df['unit'].nunique()} engines, remaining life "
      f"{test_labels_df['duration'].min():.0f} to "
      f"{test_labels_df['duration'].max():.0f} cycles")

## 4. Choose the cap for your decisions, and train

`max_rul` caps the remaining life the model learns, so it never predicts
beyond it. To choose among candidate caps, `fit()` trains each one on
cross-validation folds and scores its RMSE against the **true** remaining
life of held-out training engines.

Which forecasts it scores decides the answer. A training engine is observed
over its whole life, so most of its rows are far from failure. Scored over
all of them, the comparison is dominated by early life, where an engine with
200 or 300 cycles left needs nothing done, and a high cap wins.

`decision_horizon` says where decisions are made: the remaining life within
which you act on a forecast, roughly your lead time plus the interval
between inspections. Here it is 145 cycles, the longest remaining
life of any test engine. The caps are then compared only on forecasts made
with at most 145 cycles left. `cap_metric='rmse'` names the
comparison; it is the default and the only value so far.

The horizon changes nothing else: the other settings are tuned as usual,
and the final model is trained on all the training data with the chosen
cap and scored on the test engines, all in the one job.

In [ ]:
model = SurvivalModel({
    **SAGEMAKER,
    'model_type': 'coxph',
    # Candidate caps, compared after tuning; None means no cap.
    'max_rul': CategoricalRange([75, 100, 125, 150, 175, 200, None]),
    'decision_horizon': 145,
    'cap_metric': 'rmse',
    'calibration': 'isotonic',
    'n_trials': 12,
    'coxph_penalizer': ContinuousRange(1e-3, 10.0, log=True),
    'coxph_l1_ratio': ContinuousRange(0.0, 1.0),
    'rolling_means': CategoricalRange([[], [5, 10, 20, 30]]),
})

model.fit(train_df, val_split=0.2, test_df=test_df, test_labels_df=test_labels_df)
print(f"Chosen max_rul: {model.config['max_rul']}")

### The cap sweep within the horizon

Every candidate's RMSE, in cycles, on forecasts made with at most 145
cycles left, averaged over the five folds, as the training job returned it.
The lowest is the cap the final model was trained with.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

sweep = model.get_tuning_trials().cap_sweep
labels = ['None' if e['max_rul'] is None else str(e['max_rul']) for e in sweep]
rmse = np.array([e['rmse'] for e in sweep])
display(pd.DataFrame({'max_rul': labels, 'rmse': rmse}).set_index('max_rul').round(1).T)

fig, ax = plt.subplots(figsize=(7, 4))
for i, entry in enumerate(sweep):
    ax.scatter([i] * len(entry['fold_rmse']), entry['fold_rmse'], color='lightsteelblue', s=15)
ax.plot(range(len(rmse)), rmse, marker='o')
best = int(np.argmin(rmse))
ax.scatter([best], [rmse[best]], s=110, facecolor='none', edgecolor='C0', linewidth=2)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_xlabel('max_rul (cap on remaining life, cycles)')
ax.set_ylabel('RMSE within 145 cycles of failure')
plt.show()

## 5. Results

The final model, with the chosen cap, scored on the test engines inside the
training job. `rmse` compares with remaining life capped at the chosen
`max_rul`; `rmse_uncapped` with the true remaining life, the basis the caps
were compared on. The model is calibrated (`calibration='isotonic'`), and
these are its scores after calibration.

In [ ]:
result = model.get_evaluation_result()

print(f"max_rul: {model.config['max_rul']}")
print(f'C-index: {result.c_index:.3f}')
print(f'RMSE:    {result.rmse:.1f} (against remaining life capped at max_rul)')
print(f'         {result.rmse_uncapped:.1f} (against the true remaining life)')

## Where to go next

- **Your own horizon**: try `decision_horizon=100`. The chosen cap follows
  the horizon, though it need not equal it.
- **Your own data**: set the horizon in the same unit as `duration`, to the
  lead time you need plus the interval between inspections.

Full documentation is in the SDK repository:
https://github.com/kooditurva/lifespan-predictor-sdk